# A04 — Absorbing metal film

## Mathematical and physical model

Use time dependence $e^{-i\omega t}$, vacuum wavenumber $k_0=2\pi/\lambda_0$ and conserved transverse wavevector $k_\parallel$. For each isotropic, nonmagnetic layer set
$$
u_j=k_{z,j}/k_0=\sqrt{\epsilon_j-(k_\parallel/k_0)^2},\qquad
Y_{s,j}=u_j,\qquad Y_{p,j}=\epsilon_j/u_j .
$$
Choose the forward branch with $\operatorname{Im}u_j\ge 0$ for passive layers. The admittances apply to **tangential** electric-field coefficients. The notebook compares the solver outputs only after converting them to the common ray-based polarization and power conventions.

The film uses $\epsilon_{\mathrm{metal}}=-100+10i$ and $d=30$ nm. A complex propagation phase $\delta=k_0u_{\mathrm{metal}}d$ enters the same thin-film solution. The absorbed fraction is
$$
A=1-R-T=\frac{\omega\epsilon_0}{2P_{\mathrm{in}}}
\int_{\text{metal}}\operatorname{Im}(\epsilon_r)|\mathbf E|^2\,dV ,
$$
using the time-averaged incident power $P_{\mathrm{in}}$ over the same transverse area.

## Physical scenario

A 30 nm metal film sits between air and glass and is illuminated at normal incidence. Its complex permittivity causes absorption in addition to reflection and transmission. The benchmark uses a fixed illustrative permittivity, so the sweep isolates thin-film interference and loss handling.

## Possible applications

A starting point for modeling thin metallic mirrors, neutral-density coatings, or partially transmitting electrodes. A real device would require wavelength-dependent material data and a specified film morphology.

## Benchmark objective

Check passive absorption and phase against the thin-film reference. The metal uses a fixed illustrative near-infrared permittivity.

## Reproduce

Use the project environment and select **Run All**. The notebook and CLI use [case.yaml](case.yaml). The native [Lumerical model](model.fsp) and [reconstruction script](model.lsf) correspond to quick-sweep point 0.

[Published comparisons and figures](../../../BENCHMARK_REPORT.md#a04)

In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

from rcwa_cv.case_loader import catalog
from rcwa_cv.notebook import case_setup, show_results, show_versions, solve_sweep
from rcwa_cv.reporting.plots import geometry_figure

case_path = catalog("A04")[0]
samples, settings, metadata = case_setup(case_path)
results = {}

print(f"{len(samples)} points, Fourier truncation index {settings.order}")
figure = geometry_figure(case_path)
display(figure)
plt.close(figure)

## Analytic multilayer reference

In [ ]:
from rcwa_cv.references.multilayer import solve as solve_reference

results["reference"] = solve_sweep(solve_reference, samples, settings)

## meent · NumPy CPU

In [ ]:
from rcwa_cv.solvers.meent_cpu import probe, solve

print(probe())
results["meent"] = solve_sweep(solve, samples, settings)

## torcwa · CUDA GPU, complex128

In [ ]:
from rcwa_cv.solvers.torcwa_cuda import probe, solve

print(probe())
results["torcwa"] = solve_sweep(solve, samples, settings)

## grcwa · NumPy CPU

In [ ]:
from rcwa_cv.solvers.grcwa_cpu import probe, solve

print(probe())
results["grcwa"] = solve_sweep(solve, samples, settings)

## Lumerical RCWA

This cell uses the installed, licensed FDTD application through its Python API. It reuses one session
across the sweep and evaluates the same model generator used by the reconstruction script.

In [ ]:
from rcwa_cv.solvers.lumerical import session, solve

with session() as fdtd:
    print(f"Lumerical build {fdtd.version()}")
    results["lumerical"] = solve_sweep(solve, samples, settings)

## Compare the solvers

The spectra show total reflection, transmission and the co-polarized transmission phase.
The order plot uses a point shared by the solvers. The heatmaps show maximum discrepancies
over the tested points. Periodic quick sweeps provide preliminary comparisons.

In [ ]:
show_versions(results)
show_results(metadata, results)